# 06 — Ensemble Aggregation & Robustness Grid

Final notebook. Aggregates all model fits + validation results into headline tables and the full robustness grid per [methodology.md §6-§7](../docs/methodology.md).

**Inputs** (loaded from disk — runs independently of prior notebooks once `02-05` have populated their outputs):
- `data/results/{event}/{window}/{variant}/{model}/fit.pkl` — model fits (from 02)
- `data/validation/*.csv` — validation results (from 03-05)

**Outputs**: headline tables and ensemble plots (in-notebook), saved CSVs in `data/validation/final_*.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import DONOR_POOL_VARIANT
from lib.data import list_fits, load_fit, load_validation_table, save_validation_table
from lib.plotting import plot_ensemble_paths, plot_ensemble_gaps

MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
EVENTS = ['russia', 'hormuz']
WINDOWS = ['preferred', 'extended', 'narrow']
VARIANT = DONOR_POOL_VARIANT

print(f'Loading {len(list_fits())} fits from disk.')

Loading 30 fits from disk.
Last run: 2026-06-09 12:07:21


## Headline table — preferred specification only

Mean post-event gap (%) per model, plus ensemble median + IQR. This is the main result table for the thesis.

In [2]:
headline_rows = []
for event in EVENTS:
    event_gaps = {}
    for model in MODELS:
        fit = load_fit(event, 'preferred', model, variant=VARIANT)
        if fit is None:
            continue
        post = fit['gap'][fit['gap'].index >= fit['t0']]
        if len(post) == 0:
            continue
        mean_gap_pct = float(100 * (np.exp(post.mean()) - 1))
        event_gaps[model] = mean_gap_pct
        headline_rows.append({
            'event': event, 'model': model,
            'mean_post_gap_pct': mean_gap_pct,
            'rmspe_pre_log': fit['rmspe_pre'],
        })
    # Ensemble aggregate
    if event_gaps:
        arr = np.array(list(event_gaps.values()))
        headline_rows.append({
            'event': event, 'model': 'ENSEMBLE_MEDIAN',
            'mean_post_gap_pct': float(np.median(arr)),
            'rmspe_pre_log': np.nan,
            'iqr_lo': float(np.quantile(arr, 0.25)),
            'iqr_hi': float(np.quantile(arr, 0.75)),
            'n_models': len(arr),
        })

headline_df = pd.DataFrame(headline_rows)
save_validation_table(headline_df, 'final_headline')
headline_df.round(3)

,event,model,mean_post_gap_pct,rmspe_pre_log,iqr_lo,iqr_hi,n_models
0,russia,convex_scm,33.596,0.112,NaN,NaN,NaN
1,russia,ascm,10.874,0.112,NaN,NaN,NaN
2,russia,elastic_net,21.972,0.058,NaN,NaN,NaN
3,russia,xgboost,30.751,0.039,NaN,NaN,NaN
4,russia,bayesian_ridge,-8.989,0.046,NaN,NaN,NaN
5,russia,ENSEMBLE_MEDIAN,21.972,NaN,10.874,30.751,5.0
6,hormuz,convex_scm,58.886,0.066,NaN,NaN,NaN
7,hormuz,ascm,60.418,0.066,NaN,NaN,NaN
8,hormuz,elastic_net,60.657,0.054,NaN,NaN,NaN
9,hormuz,xgboost,54.188,0.053,NaN,NaN,NaN


Last run: 2026-06-09 12:07:21


## Robustness grid — full appendix table

Every (event, window, model) cell. Shows whether the headline estimate is stable across pre-window choices.

In [3]:
grid_rows = []
for event in EVENTS:
    for window in WINDOWS:
        for model in MODELS:
            fit = load_fit(event, window, model, variant=VARIANT)
            if fit is None:
                continue
            post = fit['gap'][fit['gap'].index >= fit['t0']]
            if len(post) == 0:
                continue
            grid_rows.append({
                'event': event, 'window': window, 'model': model,
                'mean_gap_pct': float(100 * (np.exp(post.mean()) - 1)),
                'rmspe_pre_log': fit['rmspe_pre'],
                'n_post': len(post),
            })

grid_df = pd.DataFrame(grid_rows)
save_validation_table(grid_df, 'final_robustness_grid')
grid_df.round(3)

,event,window,model,mean_gap_pct,rmspe_pre_log,n_post
0,russia,preferred,convex_scm,33.596,0.112,151
1,russia,preferred,ascm,10.874,0.112,151
2,russia,preferred,elastic_net,21.972,0.058,151
3,russia,preferred,xgboost,30.751,0.039,151
4,russia,preferred,bayesian_ridge,-8.989,0.046,151
5,russia,extended,convex_scm,51.910,0.232,151
6,russia,extended,ascm,3.164,0.232,151
7,russia,extended,elastic_net,22.566,0.118,151
8,russia,extended,xgboost,36.924,0.052,151
9,russia,extended,bayesian_ridge,-27.973,0.093,151


Last run: 2026-06-09 12:07:21


In [4]:
# Pivot for readability
pivot = grid_df.pivot_table(index=['event', 'model'], columns='window',
                             values='mean_gap_pct')
pivot = pivot.reindex(WINDOWS, axis=1)
pivot.round(2)

window                 preferred  extended  narrow
event  model                                      
hormuz ascm                60.42     46.05   54.30
       bayesian_ridge      46.46     45.50   49.21
       convex_scm          58.89     57.40   61.90
       elastic_net         60.66     56.53   56.77
       xgboost             54.19     54.59   55.64
russia ascm                10.87      3.16   32.14
       bayesian_ridge      -8.99    -27.97  -21.53
       convex_scm          33.60     51.91   32.10
       elastic_net         21.97     22.57   20.02
       xgboost             30.75     36.92   37.00

Last run: 2026-06-09 12:07:21


## Post-window sensitivity — delayed-contamination diagnostic

Per [donor_catalog.md §Temporal dimension](../docs/donor_catalog.md) (option 1). The C/M/H audit is *contemporaneous*; some donors marked clean at $T_0$ may acquire a Russia component **over** the post-window (fertilizer cost-push into the soft commodities, discounted-crude re-routing into INR/CNY, terms-of-trade into commodity FX, the inflation→rates path into TLT/HYG). Such **delayed contamination** enters the post-window *projection* — not the pre-window fit — so it pulls the synthetic **up** and biases the gap **toward zero**, with the bias growing as the horizon lengthens.

This re-summarises the **same** preferred-window fits over progressively longer post-windows (1, 2, 3, 6 months, and full). No refitting: weights are learned pre-$T_0$ and unchanged; only the horizon over which the gap is averaged changes. Reading:

- **Monotone decline** with horizon → delayed-contamination signature; the short-horizon estimate is the least-contaminated bound.
- **Flat** → delayed contamination is immaterial; the headline gap is horizon-robust.
- **Rising** → the treatment effect is still building in (the opposite of contamination).

This is the post-window complement to the pre-window robustness grid above, and shares the logic of the OPEC+ truncation ([methodology.md §2](../docs/methodology.md)). Output: `data/validation/final_postwindow_sensitivity.csv`.

In [5]:
# ===== Post-window sensitivity (donor_catalog.md §Temporal dimension, option 1) =====
# Delayed/cumulative donor contamination enters the POST-window projection and biases
# the gap toward zero, worsening with horizon. Re-summarise the SAME pre-window fits
# over progressively longer post-windows -- no refitting (weights are learned pre-T0).
from pandas.tseries.offsets import DateOffset

MONTH_HORIZONS = [1, 2, 3, 6]

sens_rows = []
for event in EVENTS:
    fits = {m: load_fit(event, 'preferred', m, variant=VARIANT) for m in MODELS}
    fits = {m: f for m, f in fits.items() if f is not None}
    if not fits:
        continue
    t0 = next(iter(fits.values()))['t0']
    ref_idx = next(iter(fits.values()))['gap'].index
    full_end = max(f['gap'].index.max() for f in fits.values())

    horizons = [(f'{h}m', t0 + DateOffset(months=h)) for h in MONTH_HORIZONS
                if t0 + DateOffset(months=h) < full_end]
    horizons.append(('full', full_end))

    for label, end in horizons:
        model_gaps = {}
        for m, f in fits.items():
            post = f['gap'][(f['gap'].index >= t0) & (f['gap'].index <= end)]
            if len(post):
                model_gaps[m] = float(100 * (np.exp(post.mean()) - 1))
        arr = np.array(list(model_gaps.values()))
        n_post = int(((ref_idx >= t0) & (ref_idx <= end)).sum())
        row = {'event': event, 'horizon': label, 'horizon_end': str(end.date()),
               'n_post': n_post,
               'ens_median_gap_pct': float(np.median(arr)),
               'iqr_lo': float(np.quantile(arr, 0.25)),
               'iqr_hi': float(np.quantile(arr, 0.75))}
        for m in MODELS:
            row[m] = model_gaps.get(m, np.nan)
        sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows)
save_validation_table(sens_df, 'final_postwindow_sensitivity')

# Diagnosis: compare shortest vs full ensemble median per event.
print('Post-window sensitivity -- ensemble-median gap (%) by horizon:\n')
for event in EVENTS:
    sub = sens_df[sens_df['event'] == event]
    seq = sub['ens_median_gap_pct'].values
    drop = seq[0] - seq[-1]
    if drop > 3:
        verdict = (f'DECLINING by {drop:.1f} pp from {sub.iloc[0]["horizon"]} to full '
                   f'-> delayed-contamination signature (gap attenuates with horizon; '
                   f'short-horizon = least-contaminated bound)')
    elif drop < -3:
        verdict = (f'RISING by {-drop:.1f} pp -> effect still building in '
                   f'(no attenuation; opposite of contamination)')
    else:
        verdict = (f'FLAT (delta={drop:+.1f} pp) -> delayed contamination immaterial; '
                   f'headline gap robust to horizon')
    print(f'{event}: ' + '  '.join(f'{r.horizon}={r.ens_median_gap_pct:.1f}%'
                                   for r in sub.itertuples()))
    print(f'    -> {verdict}\n')

sens_df.round(2)

Post-window sensitivity -- ensemble-median gap (%) by horizon:

russia: 1m=31.4%  2m=24.4%  3m=25.9%  6m=27.5%  full=22.0%
    -> DECLINING by 9.4 pp from 1m to full -> delayed-contamination signature (gap attenuates with horizon; short-horizon = least-contaminated bound)

hormuz: 1m=46.2%  2m=59.1%  3m=59.3%  full=58.9%
    -> RISING by 12.7 pp -> effect still building in (no attenuation; opposite of contamination)



,event,horizon,horizon_end,n_post,ens_median_gap_pct,iqr_lo,iqr_hi,convex_scm,ascm,elastic_net,xgboost,bayesian_ridge
0,russia,1m,2022-03-24,21,31.37,29.18,36.94,40.68,31.37,36.94,29.18,24.07
1,russia,2m,2022-04-24,40,24.36,23.90,29.94,35.00,23.90,29.94,24.36,13.62
2,russia,3m,2022-05-24,61,25.88,21.43,29.24,35.27,21.43,29.24,25.88,10.80
3,russia,6m,2022-08-24,126,27.53,17.08,33.58,38.04,17.08,27.53,33.58,-0.46
4,russia,full,2022-09-30,151,21.97,10.87,30.75,33.60,10.87,21.97,30.75,-8.99
5,hormuz,1m,2026-03-28,20,46.17,42.63,49.18,46.17,49.18,50.44,42.63,39.07
6,hormuz,2m,2026-04-28,40,59.07,54.10,61.37,59.07,61.37,61.66,54.10,47.63
7,hormuz,3m,2026-05-28,60,59.27,54.58,60.87,59.27,60.87,61.14,54.58,46.86
8,hormuz,full,2026-05-29,61,58.89,54.19,60.42,58.89,60.42,60.66,54.19,46.46


Last run: 2026-06-09 12:07:21


In [6]:
import plotly.graph_objects as go
from lib.plotting import save_html

for event in EVENTS:
    sub = sens_df[sens_df['event'] == event].reset_index(drop=True)
    x = sub['horizon'].tolist()
    lo, hi = sub['iqr_lo'].tolist(), sub['iqr_hi'].tolist()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x + x[::-1], y=hi + lo[::-1], fill='toself',
                             fillcolor='rgba(31,119,180,0.15)', line=dict(width=0),
                             name='IQR (model spread)', hoverinfo='skip'))
    for m in MODELS:
        if m in sub.columns:
            fig.add_trace(go.Scatter(x=x, y=sub[m], mode='lines', name=m,
                                     line=dict(width=1, dash='dot'), opacity=0.5))
    fig.add_trace(go.Scatter(x=x, y=sub['ens_median_gap_pct'], mode='lines+markers',
                             name='ensemble median', line=dict(color='#1f77b4', width=3)))
    fig.update_layout(title=f'{event.title()} — post-window sensitivity of the gap '
                            f'(declining = delayed-contamination signature)',
                      xaxis_title='post-window horizon', yaxis_title='mean gap (%)',
                      template='plotly_white', height=430,
                      margin=dict(t=70, b=40, l=60, r=40))
    save_html(fig, f'{event}_postwindow_sensitivity', ROOT / 'plots').show()

Last run: 2026-06-09 12:07:21


## Contamination-bias decomposition: weight the signed divergence by model importance

Combines Battery D's per-donor delta_j (01_5) with each model's fitted weights to get the net signed ATT bias per model per horizon:   
- B = -sum_j w_j * delta_j

B < 0  -> net UNDERestimate (conservative; the teammates' "toward zero" claim holds)

B > 0  -> net OVERestimate  (crowd-out dominates; the supervisor's concern realised)

Exact for the four LINEAR models (synthetic = sum_j w_j * donor_level). XGBoost is nonlinear so its w_j are importance magnitudes, not signed coefficients -> reported but EXCLUDED from B.

In [7]:
LINEAR_MODELS = ['convex_scm', 'ascm', 'elastic_net', 'bayesian_ridge']
HORIZON_LABELS = ['1w', '1m', '3m', 'full']

bias_rows, contrib_rows = [], []
for event in EVENTS:
    div = load_validation_table(f'donor_signed_divergence_{event}')
    if div is None:
        print(f'{event}: run Battery D in 01_5 first (no donor_signed_divergence table).')
        continue
    div = div.set_index('donor')

    for model in MODELS:
        fit = load_fit(event, 'preferred', model, variant=VARIANT)
        if fit is None:
            continue
        w = fit['weights'].reindex(div.index).fillna(0.0)
        w_abs_sum = w.abs().sum() or 1.0
        is_linear = model in LINEAR_MODELS

        for h in HORIZON_LABELS:
            delta = div[f'delta_{h}']
            contrib = -(w * delta)                        # per-donor bias contribution
            B = float(contrib.sum()) if is_linear else np.nan
            row = {'event': event, 'model': model, 'horizon': h,
                   'net_bias_log': B,
                   'net_bias_pct_approx': float(100 * (np.exp(B) - 1)) if is_linear else np.nan,
                   'verdict': ('underestimate (conservative)' if is_linear and B < 0
                               else 'overestimate' if is_linear and B > 0
                               else 'n/a (nonlinear)')}
            # Which donors push each way (top contributors by |contribution|), full-window only.
            if h == 'full':
                top = contrib.reindex(w[w.abs() > 0.01].index).dropna().abs().sort_values(ascending=False)
                for d in top.head(4).index:
                    contrib_rows.append({'event': event, 'model': model, 'donor': d,
                                         'weight': round(float(w[d]), 4),
                                         'delta_full': round(float(div.loc[d, 'delta_full']), 4),
                                         'bias_contrib': round(float(contrib[d]), 5),
                                         'pushes': 'toward underestimate' if contrib[d] < 0 else 'toward overestimate'})
            bias_rows.append(row)

bias_df = pd.DataFrame(bias_rows)
contrib_df = pd.DataFrame(contrib_rows)
save_validation_table(bias_df, 'contamination_bias_decomposition')
save_validation_table(contrib_df, 'contamination_bias_top_contributors')

print('Net signed ATT bias  B = -sum_j w_j*delta_j   (linear models; negative = conservative):\n')
for event in EVENTS:
    sub = bias_df[(bias_df.event == event) & bias_df.net_bias_log.notna()]
    if sub.empty:
        continue
    piv = sub.pivot(index='model', columns='horizon', values='net_bias_pct_approx')[HORIZON_LABELS]
    print(f'{event} — net bias as approx % of the gap, by model x horizon:')
    print(piv.round(2).to_string())
    full = sub[sub.horizon == 'full']
    n_under = (full.net_bias_log < 0).sum(); n_over = (full.net_bias_log > 0).sum()
    print(f'  full-window: {n_under}/{len(full)} linear models net-conservative (underestimate), '
          f'{n_over} net-overestimate\n')

bias_df.round(4)

Net signed ATT bias  B = -sum_j w_j*delta_j   (linear models; negative = conservative):

russia — net bias as approx % of the gap, by model x horizon:
horizon           1w     1m     3m   full
model                                    
ascm            8.96   9.56  10.51  12.69
bayesian_ridge  7.91   4.47   1.40  -7.15
convex_scm      4.57   7.40  11.56  21.42
elastic_net     7.48  10.49  14.14  20.60
  full-window: 1/4 linear models net-conservative (underestimate), 3 net-overestimate

hormuz — net bias as approx % of the gap, by model x horizon:
horizon           1w    1m    3m  full
model                                 
ascm            0.82 -1.59 -2.11 -2.11
bayesian_ridge -1.38 -3.36 -7.59 -7.59
convex_scm      0.94 -1.54 -1.28 -1.28
elastic_net    -0.16 -0.97 -2.40 -2.40
  full-window: 4/4 linear models net-conservative (underestimate), 0 net-overestimate



,event,model,horizon,net_bias_log,net_bias_pct_approx,verdict
0,russia,convex_scm,1w,0.0447,4.5682,overestimate
1,russia,convex_scm,1m,0.0714,7.4047,overestimate
2,russia,convex_scm,3m,0.1094,11.5629,overestimate
3,russia,convex_scm,full,0.1941,21.4202,overestimate
4,russia,ascm,1w,0.0858,8.9615,overestimate
5,russia,ascm,1m,0.0913,9.5616,overestimate
6,russia,ascm,3m,0.1000,10.5124,overestimate
7,russia,ascm,full,0.1194,12.6872,overestimate
8,russia,elastic_net,1w,0.0721,7.4798,overestimate
9,russia,elastic_net,1m,0.0997,10.4887,overestimate


Last run: 2026-06-09 12:07:21


### High-R² restriction — separating contamination from idiosyncratic-proxy drift

The full bias `B = -sum_j w_j*delta_j` mixes two things: donors whose post-window divergence is
attributable to the common macro factor (plausibly **event contamination**, SUTVA-relevant) and
donors that are simply **noisy proxies** decoupled from the factor (e.g. Coffee's 2022 supply story —
not Russia). The pre-window factor `R²` separates them. We split each model's bias additively:

`B_full = B_highR2 (reliable / contamination) + B_lowR2 (idiosyncratic-proxy drift)`

and report each bias as a **share of that model's own gap** at the same horizon, so the magnitude is
read against the effect it is meant to qualify.

In [8]:
# ===== High-R² restriction + bias-as-share-of-gap (validation.md §5g robustness) =====
# Split B into reliable (high-R^2) vs idiosyncratic (low-R^2) parts, and express both as a
# share of each model's own gap. R2_FLOOR cleanly separates the top ~8 donors per event;
# try 0.05 / 0.10 to confirm the DIRECTION is not knife-edge on the cutoff.
R2_FLOOR = 0.08
LINEAR_MODELS = ['convex_scm', 'ascm', 'elastic_net', 'bayesian_ridge']
HZ = [('1w', 5), ('1m', 21), ('3m', 63), ('full', None)]   # same bday horizons as Battery D

def _gap_log_at(fit, t0, hbd):
    post = fit['gap'][fit['gap'].index >= t0]
    seg = post if hbd is None else post.iloc[:hbd]
    return float(seg.mean()) if len(seg) else np.nan

hi_rows = []
for event in EVENTS:
    div = load_validation_table(f'donor_signed_divergence_{event}')
    if div is None:
        print(f'{event}: run Battery D in 01_5 first.'); continue
    div = div.set_index('donor')
    reliable = div.index[div['pre_R2'] >= R2_FLOOR]

    for model in LINEAR_MODELS:
        fit = load_fit(event, 'preferred', model, variant=VARIANT)
        if fit is None:
            continue
        w = fit['weights'].reindex(div.index).fillna(0.0)
        w_total = w.abs().sum() or np.nan
        coverage = float(w.reindex(reliable).abs().sum() / w_total)
        t0 = fit['t0']

        for label, hbd in HZ:
            delta = div[f'delta_{label}']
            contrib = -(w * delta)                                  # per-donor bias contribution
            B_full = float(contrib.sum())
            B_hi   = float(contrib.reindex(reliable).sum())
            B_lo   = B_full - B_hi
            gap_log = _gap_log_at(fit, t0, hbd)
            to_pct = lambda b: 100 * (np.exp(b) - 1)
            hi_rows.append({
                'event': event, 'model': model, 'horizon': label,
                'gap_pct':          to_pct(gap_log),
                'B_full_pct':       to_pct(B_full),
                'B_highR2_pct':     to_pct(B_hi),
                'B_lowR2_pct':      to_pct(B_lo),
                'highR2_wt_coverage': coverage,
                'share_of_gap_full':   (B_full / gap_log) if gap_log else np.nan,
                'share_of_gap_highR2': (B_hi   / gap_log) if gap_log else np.nan,
            })

hi_df = pd.DataFrame(hi_rows)
save_validation_table(hi_df, 'contamination_bias_highR2')

# --- (A) model-agnostic: do the RELIABLE donors agree on direction? ---
print(f'Reliable donors (pre_R2 >= {R2_FLOOR}) — direction of delta_full:\n')
for event in EVENTS:
    div = load_validation_table(f'donor_signed_divergence_{event}').set_index('donor')
    rel = div[div['pre_R2'] >= R2_FLOOR].sort_values('pre_R2', ascending=False)
    up = (rel['delta_full'] > 0).sum(); dn = (rel['delta_full'] < 0).sum()
    print(f'{event}: {len(rel)} reliable donors -> {up} UP / {dn} DOWN')
    print(rel[['pre_R2', 'delta_full']].round(3).to_string()); print()

# --- (B) full-window: how much of each model's gap is bias, and how much is reliable? ---
print('Full-window decomposition (positive = overestimate; share = bias / gap):\n')
show = ['gap_pct', 'B_full_pct', 'B_highR2_pct', 'B_lowR2_pct',
        'highR2_wt_coverage', 'share_of_gap_full', 'share_of_gap_highR2']
for event in EVENTS:
    sub = hi_df[(hi_df.event == event) & (hi_df.horizon == 'full')].set_index('model')[show]
    print(f'{event}:'); print(sub.round(3).to_string()); print()

hi_df.round(3)

Reliable donors (pre_R2 >= 0.08) — direction of delta_full:

russia: 5 reliable donors -> 0 UP / 5 DOWN
          pre_R2  delta_full
donor                       
AUD        0.170      -0.023
Silver     0.105      -0.230
Platinum   0.093      -0.239
SP500      0.092      -0.109
Gold       0.084      -0.076

hormuz: 6 reliable donors -> 2 UP / 4 DOWN
          pre_R2  delta_full
donor                       
Silver     0.168      -0.246
Platinum   0.154      -0.205
HYG        0.128      -0.012
Gold       0.116      -0.156
SP500      0.102       0.004
JPY        0.095       0.024

Full-window decomposition (positive = overestimate; share = bias / gap):

russia:
                gap_pct  B_full_pct  B_highR2_pct  B_lowR2_pct  highR2_wt_coverage  share_of_gap_full  share_of_gap_highR2
model                                                                                                                     
convex_scm       33.596      21.420         0.738       20.531               0.031      

,event,model,horizon,gap_pct,B_full_pct,B_highR2_pct,B_lowR2_pct,highR2_wt_coverage,share_of_gap_full,share_of_gap_highR2
0,russia,convex_scm,1w,28.701,4.568,0.143,4.419,0.031,0.177,0.006
1,russia,convex_scm,1m,40.683,7.405,0.169,7.224,0.031,0.209,0.005
2,russia,convex_scm,3m,35.539,11.563,0.446,11.067,0.031,0.360,0.015
3,russia,convex_scm,full,33.596,21.420,0.738,20.531,0.031,0.670,0.025
4,russia,ascm,1w,23.400,8.961,3.229,5.553,0.295,0.408,0.151
5,russia,ascm,1m,31.365,9.562,4.285,5.060,0.295,0.335,0.154
6,russia,ascm,3m,21.548,10.512,6.782,3.493,0.295,0.512,0.336
7,russia,ascm,full,10.874,12.687,5.159,7.158,0.295,1.157,0.487
8,russia,elastic_net,1w,26.141,7.480,2.374,4.988,0.355,0.311,0.101
9,russia,elastic_net,1m,36.943,10.489,3.327,6.931,0.355,0.317,0.104


Last run: 2026-06-09 12:07:21


## Naïve pre-trend baselines — the deliberately-contaminated benchmark

A univariate counterfactual built from Brent's OWN pre-window series (no donors): random walk
(flat), random-walk-with-drift, linear pre-trend, and two **ARIMA frozen-forecast** counterfactuals.
These are "contaminated" by design — they cannot remove the common macro factors the SCM strips out,
and they extrapolate any pre-event trend forward. Purpose: (i) an added-value floor for the SCM,
(ii) a transparent reference reviewers can compute by hand, and (iii) an independent read on the §5g
sign question — `SCM gap − naïve gap` is the common-factor adjustment, and its sign should
corroborate §5g without using any donor.

The two ARIMA lines are an **interrupted-time-series (ITS) counterfactual built on the best ARIMA
model** — fit on the pre-window *only*, parameters frozen, then dynamically forecast over the
post-window with **no updating**. `arima_rw` is the honest best-AIC ARIMA(1,0,0) with stationarity
enforced (φ≈1 ⇒ a random walk, so ≈ `rw_flat`); the ITS effect is simply `observed − forecast`.
`arima_mr` imposes a 6-month mean-reversion half-life (an assumption, shown only for sensitivity).
NOT part of the SCM ensemble.

In [9]:
# ===== Naive univariate baselines on log-Brent (no donors) =====
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from lib.data import build_panel

HZ = [('1w', 5), ('1m', 21), ('3m', 63), ('full', None)]   # same bday horizons as section 5g
ARIMA_HALFLIFE_BD = 126   # imposed OU mean-reversion half-life for arima_mr (~6 trading months)

def _naive_counterfactuals(logbrent, t0):
    """Univariate counterfactuals on log-Brent, projected over the post-window."""
    pre  = logbrent[logbrent.index < t0]
    post = logbrent[logbrent.index >= t0]
    n_pre, n_post = len(pre), len(post)
    last = pre.iloc[-1]
    cf = {}
    cf['rw_flat']  = pd.Series(last, index=post.index)                              # carry last value
    drift = pre.diff().dropna().mean()
    cf['rw_drift'] = pd.Series(last + drift * np.arange(1, n_post + 1), index=post.index)
    slope, intercept = np.polyfit(np.arange(n_pre), pre.values, 1)                  # OLS log-Brent on time
    cf['lin_trend'] = pd.Series(intercept + slope * np.arange(n_pre, n_pre + n_post), index=post.index)

    # ----- ARIMA-based interrupted-time-series (ITS) counterfactuals: fit on the PRE-window, freeze the
    #       parameters, then run a dynamic multi-step forecast over the post-window (NO post observations
    #       fed in). The ITS effect is observed - frozen forecast.
    # A full SARIMA grid by AIC selects ARIMA(1,0,0) -- an AR(1) on log-price with phi ~ 1.0, i.e. a
    # random walk; no seasonal term is ever justified on daily Brent. We report TWO frozen forecasts:
    #   arima_rw : best-AIC AR(1), stationarity enforced (phi <= 1) -> the HONEST near-flat RW ITS.
    #   arima_mr : AR(1)/Ornstein-Uhlenbeck with an IMPOSED mean-reversion half-life -> a curved forecast
    #              that decays from the last pre-value toward the pre-window mean. The MLE phi is ~1.000,
    #              so this reversion is an ASSUMPTION, not a data finding (see docs/methodology.md).
    tr = 'n' if slope > 0 else 'c'
    arima_info = {'order': (1, 0, 0), 'trend': tr, 'halflife_bd': ARIMA_HALFLIFE_BD}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res = SARIMAX(pre.values, order=(1, 0, 0), trend=tr,
                          enforce_stationarity=True, enforce_invertibility=False).fit(disp=0, maxiter=300)
        cf['arima_rw'] = pd.Series(np.asarray(res.get_forecast(steps=n_post).predicted_mean), index=post.index)
        arima_info.update(phi=float(res.arparams[0]) if len(res.arparams) else np.nan, aic=float(res.aic))
    except Exception:
        cf['arima_rw'] = cf['rw_flat'].copy()
        arima_info.update(phi=np.nan, aic=np.nan)
    mean_log = pre.mean()
    phi_mr = 0.5 ** (1.0 / ARIMA_HALFLIFE_BD)
    h = np.arange(1, n_post + 1)
    cf['arima_mr'] = pd.Series(mean_log + (last - mean_log) * phi_mr ** h, index=post.index)
    arima_info['mr_mean_usd'] = float(np.exp(mean_log))
    return cf, post, slope, arima_info

def _gap_pct_at(obs_log, cf_log, hbd):
    g = (obs_log - cf_log).dropna()
    seg = g if hbd is None else g.iloc[:hbd]
    return float(100 * (np.exp(seg.mean()) - 1)) if len(seg) else np.nan

# arima_rw / arima_mr are frozen-forecast ARIMA counterfactuals (fit pre, no post updating).
# arima_rw IS the ARIMA-based interrupted-time-series (ITS): ITS effect = observed - frozen forecast.
NAIVE = ['rw_flat', 'rw_drift', 'lin_trend', 'arima_rw', 'arima_mr']
base_rows, naive_cf_store = [], {}
for event in EVENTS:
    panel, meta = build_panel(event=event, window='preferred', variant=VARIANT)
    t0 = meta['t0']
    obs_log = panel['Brent']
    cf, post, pre_slope, arima_info = _naive_counterfactuals(obs_log, t0)
    naive_cf_store[event] = (obs_log, cf, t0, meta, pre_slope, arima_info)

    fits = {m: load_fit(event, 'preferred', m, variant=VARIANT) for m in MODELS}
    fits = {m: f for m, f in fits.items() if f is not None}
    for label, hbd in HZ:
        scm_each = []
        for f in fits.values():
            pp = f['gap'][f['gap'].index >= t0]
            seg = pp if hbd is None else pp.iloc[:hbd]
            if len(seg):
                scm_each.append(100 * (np.exp(seg.mean()) - 1))
        row = {'event': event, 'horizon': label,
               'scm_median': float(np.median(scm_each)) if scm_each else np.nan}
        for nm in NAIVE:
            row[nm] = _gap_pct_at(obs_log[obs_log.index >= t0], cf[nm], hbd)
        row['scm_minus_rwflat']   = row['scm_median'] - row['rw_flat']    # vs trend-free null (featured)
        row['scm_minus_rwdrift']  = row['scm_median'] - row['rw_drift']
        row['scm_minus_lintrend'] = row['scm_median'] - row['lin_trend']
        row['scm_minus_arimarw']  = row['scm_median'] - row['arima_rw']   # vs ARIMA-based ITS counterfactual
        row['scm_minus_arimamr']  = row['scm_median'] - row['arima_mr']
        base_rows.append(row)

base_df = pd.DataFrame(base_rows)
save_validation_table(base_df, 'naive_baseline_comparison')

print('Gap (%) -- SCM ensemble median vs naive univariate baselines (incl. ARIMA-based ITS):\n')
cols = ['scm_median', 'rw_flat', 'rw_drift', 'lin_trend', 'arima_rw', 'arima_mr',
        'scm_minus_rwflat', 'scm_minus_arimarw']
for event in EVENTS:
    sub = base_df[base_df.event == event].set_index('horizon')[cols]
    _, _, _, _, pre_slope, arima_info = naive_cf_store[event]
    trend = 'UP-trend (rally)' if pre_slope > 0 else 'DOWN-trend'
    print(f'{event}  [pre-window {trend}]:')
    print(sub.round(2).to_string())
    full = sub.loc['full']
    print(f'  full vs trend-free null (rw_flat): SCM {full["scm_median"]:.1f}% vs {full["rw_flat"]:.1f}% '
          f'({full["scm_minus_rwflat"]:+.1f} pp) -> SCM attributes '
          f'{"MORE" if full["scm_minus_rwflat"] > 0 else "LESS"} to the event than a no-change null.')
    print(f'  ARIMA-based ITS: best-AIC model = ARIMA{arima_info["order"]} trend={arima_info["trend"]} '
          f'(phi={arima_info.get("phi", float("nan")):.4f}, AIC={arima_info.get("aic", float("nan")):.1f}). '
          f'phi~1 => the frozen forecast (arima_rw) is ~flat (random walk), so the ITS effect '
          f'(observed - forecast) = {full["arima_rw"]:.1f}% at the full horizon. arima_mr imposes a '
          f'{arima_info["halflife_bd"]}-bday reversion half-life toward pre-window mean '
          f'${arima_info["mr_mean_usd"]:.1f} (assumption, not MLE).')
    print()

base_df.round(2)


Gap (%) -- SCM ensemble median vs naive univariate baselines (incl. ARIMA-based ITS):

russia  [pre-window UP-trend (rally)]:
         scm_median  rw_flat  rw_drift  lin_trend  arima_rw  arima_mr  scm_minus_rwflat  scm_minus_arimarw
horizon                                                                                                   
1w            23.40     7.07      6.41       9.91      7.09      7.89             16.33              16.31
1m            31.37    16.01     13.43      17.12     16.08     19.22             15.36              15.28
3m            26.33    11.59      4.54       7.84     11.80     20.15             14.74              14.52
full          21.97     8.70     -6.90      -4.15      9.20     26.50             13.27              12.78
  full vs trend-free null (rw_flat): SCM 22.0% vs 8.7% (+13.3 pp) -> SCM attributes MORE to the event than a no-change null.
  ARIMA-based ITS: best-AIC model = ARIMA(1, 0, 0) trend=n (phi=1.0000, AIC=-2031.2). phi~1 => the frozen f

,event,horizon,scm_median,rw_flat,rw_drift,lin_trend,arima_rw,arima_mr,scm_minus_rwflat,scm_minus_rwdrift,scm_minus_lintrend,scm_minus_arimarw,scm_minus_arimamr
0,russia,1w,23.40,7.07,6.41,9.91,7.09,7.89,16.33,16.98,13.49,16.31,15.51
1,russia,1m,31.37,16.01,13.43,17.12,16.08,19.22,15.36,17.93,14.25,15.28,12.14
2,russia,3m,26.33,11.59,4.54,7.84,11.80,20.15,14.74,21.78,18.49,14.52,6.18
3,russia,full,21.97,8.70,-6.90,-4.15,9.20,26.50,13.27,28.87,26.12,12.78,-4.53
4,hormuz,1w,26.54,19.25,19.31,34.30,19.16,19.24,7.29,7.24,-7.76,7.38,7.31
5,hormuz,1m,47.39,41.83,42.07,60.45,41.49,41.77,5.56,5.31,-13.06,5.89,5.61
6,hormuz,3m,58.89,51.89,52.64,73.77,51.12,51.74,6.99,6.25,-14.88,7.76,7.14
7,hormuz,full,58.89,51.89,52.64,73.77,51.12,51.74,6.99,6.25,-14.88,7.76,7.14


Last run: 2026-06-09 12:07:22


In [10]:
# ===== Counterfactual paths: observed Brent vs SCM synthetic vs naive baselines =====
import plotly.graph_objects as go
from lib.plotting import save_html, PALETTE

for event in EVENTS:
    obs_log, cf, t0, meta, pre_slope, arima_info = naive_cf_store[event]   # 6-tuple
    fits = {m: load_fit(event, 'preferred', m, variant=VARIANT) for m in MODELS}
    fits = {m: f for m, f in fits.items() if f is not None}
    full_idx = obs_log.index

    # SCM ensemble-mean synthetic (single clean line in USD/bbl)
    synth_lvls = pd.concat([np.exp(f['synth']) for f in fits.values()], axis=1).mean(axis=1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=full_idx, y=np.exp(obs_log).values, name='Observed Brent',
                             line=dict(color=PALETTE['actual'], width=2.5)))
    fig.add_trace(go.Scatter(x=synth_lvls.index, y=synth_lvls.values, name='SCM synthetic (ens. mean)',
                             line=dict(color=PALETTE['synth'], width=1.8, dash='dash')))
    style = {'lin_trend': ('#2ca02c', 'linear pre-trend'),
             'rw_drift':  ('#9467bd', 'RW + drift'),
             'rw_flat':   ('#ff7f0e', 'ARIMA(0,1,0)'),
             'arima_rw':  ('#8c564b', 'ARIMA(1,0,0)'),
             'arima_mr':  ('#e377c2', 'ARIMA mean-reverting')}
    for nm, (color, lbl) in style.items():
        dash = 'longdash' if nm.startswith('arima') else 'dot'
        fig.add_trace(go.Scatter(x=cf[nm].index, y=np.exp(cf[nm]).values, name=lbl,
                                 line=dict(color=color, width=1.4, dash=dash)))
    fig.add_vline(x=t0, line_dash='dot', line_color='grey')
    fig.update_layout(title=f'{event.title()} -- observed vs SCM synthetic vs naive counterfactuals',
                      yaxis_title='USD / bbl', xaxis_title='Date', template='plotly_white',
                      hovermode='x unified', height=470,
                      legend=dict(orientation='h', y=-0.22, x=0))
    save_html(fig, f'{event}_naive_baseline_comparison', ROOT / 'plots').show()


Last run: 2026-06-09 12:07:22


## Ensemble visualization — preferred window

In [11]:
for event in EVENTS:
    fits = {}
    for model in MODELS:
        f = load_fit(event, 'preferred', model, variant=VARIANT)
        if f is not None:
            fits[model] = f
    if fits:
        plot_ensemble_paths(fits, title=f'{event.title()} — ensemble synthetic paths').show()
        plot_ensemble_gaps(fits, title=f'{event.title()} — ensemble gap (%)').show()

Last run: 2026-06-09 12:07:22


## Validation pass/fail summary (from 03_Validate)

In [12]:
valid_summary = load_validation_table('validation_summary')
if valid_summary is not None:
    print(valid_summary.round(4).to_string())
else:
    print('validation_summary.csv not found — run 03_Validate first.')

    event           model  wf_train_rmse  wf_val_rmse  wf_ratio  pf_mean_pct  pf_slope_yr   pf_r2  drift_contribution_pct
0  russia      convex_scm         0.1131       0.1412    1.2483       0.6152       8.9364  0.1491                  5.1831
1  russia            ascm         0.0458       0.0946    2.0646       0.1336       0.5180  0.0023                  0.3004
2  russia     elastic_net         0.0509       0.1016    1.9941       0.1676       1.2569  0.0107                  0.7290
3  russia         xgboost         0.0391       0.1296    3.3118       0.0753       3.1753  0.1567                  1.8417
4  russia  bayesian_ridge         0.0338       0.1045    3.0886       0.1038       0.2004  0.0004                  0.1162
5  hormuz      convex_scm         0.0597       0.0833    1.3952       0.2589      -5.4267  0.1681                 -1.3567
6  hormuz            ascm         0.0488       0.0751    1.5402       0.1621      -1.5094  0.0189                 -0.3773
7  hormuz     elastic_ne

## Cross-event transfer table (from 05_Cross_Event)

In [13]:
transfer = load_validation_table('cross_event_transfer')
if transfer is not None:
    print(transfer.round(3).to_string())
else:
    print('cross_event_transfer.csv not found — run 05_Cross_Event first.')

            model  russia_rmspe_pre  hormuz_independent_rmspe_pre  hormuz_transferred_rmspe_pre  hormuz_independent_post_gap_pct  hormuz_transferred_post_gap_pct  transferred_minus_independent_pct
0      convex_scm             0.112                         0.066                         0.336                           58.887                           25.914                            -32.973
1            ascm             0.112                         0.066                         0.417                           56.341                            4.807                            -51.534
2     elastic_net             0.058                         0.052                         2.917                           61.263                          -92.666                           -153.929
3         xgboost             0.039                         0.053                         0.956                           54.188                          -44.382                            -98.571
4  bayesian_rid

## External validation — EIA STEO pre-invasion forecasts (Russia only)

Compare the SCM synthetic counterfactual against the U.S. EIA's last pre-invasion Brent forecasts. Both STEOs were issued *before* the 2022-02-24 invasion, so their forecasts for the post-event window represent EIA's independent counterfactual built on a structural supply/demand model:

- **STEO Jan-22** issued ~2022-01-11 (six weeks pre-invasion)
- **STEO Feb-22** issued ~2022-02-08 (sixteen days pre-invasion, the last STEO before the invasion)

The two methods (SCM cross-asset co-movement vs STEO supply/demand structural model) share no information path. Agreement is evidence that the SCM is not producing a fantasy counterfactual; disagreement is informative about which factor structure each method emphasises.

**Source**: STEO archives at <https://www.eia.gov/outlooks/steo/archives/> (publicly accessible XLSX files). Files **tracked in `data/external/eia/`** — committed verbatim from EIA so provenance is captured even if EIA reorganises their archive. Filenames in the repo match the EIA archive URL filenames (`jan22_base.xlsx`, `feb22_base.xlsx`) for trivial verification. The Brent row is `BREPUUS` (EIA's standard variable code for Brent spot price) in worksheet `2tab`; 2022 monthly values occupy columns 50–61 (the 48-month offset from Jan-2018 baseline).

In [14]:
"""External validation: SCM synthetic vs EIA STEO pre-invasion forecasts (Russia 2022).

Compare the SCM ensemble counterfactual against EIA's two last pre-invasion
forecasts of 2022 Brent. The two methods share no information path (SCM uses
cross-asset co-movement; STEO uses an internal supply/demand model), so
agreement is evidence the SCM is not producing a fantasy counterfactual.

Method: build a daily step-function from each STEO's monthly forecast values
(each daily observation in month M takes the STEO's forecast for M), then
take the mean over the exact SCM treatment day-set (Feb 24 - Sep 30 2022,
151 trading days).

Source files: tracked in data/external/eia/{jan22,feb22}_base.xlsx, copied
verbatim from https://www.eia.gov/outlooks/steo/archives/ (filenames match
the EIA archive URLs so provenance is trivially verifiable). If the tracked
files are missing, the script falls back to downloading from EIA."""
import urllib.request
import openpyxl
import plotly.graph_objects as go
from lib.plotting import save_html

EIA_DIR = ROOT / 'data' / 'external' / 'eia'   # tracked; files committed to repo
EIA_DIR.mkdir(parents=True, exist_ok=True)
STEO_URLS = {
    'jan22': 'https://www.eia.gov/outlooks/steo/archives/jan22_base.xlsx',
    'feb22': 'https://www.eia.gov/outlooks/steo/archives/feb22_base.xlsx',
}

def _load_steo_brent_2022(tag):
    """Return STEO monthly Brent spot forecast for 2022 (12 values, Jan-Dec).

    Filename in repo matches the EIA archive URL filename so anyone can
    verify by downloading the same file directly from EIA.
    """
    p = EIA_DIR / f'{tag}_base.xlsx'
    if not p.exists():
        print(f'  {p.name} not in repo, downloading from {STEO_URLS[tag]}')
        urllib.request.urlretrieve(STEO_URLS[tag], p)
    wb = openpyxl.load_workbook(p, data_only=True)
    for row in wb['2tab'].iter_rows(min_row=1, max_row=40, values_only=True):
        if row[0] == 'BREPUUS':
            # data starts at row[2] = 2018-01; 2022 = +48 months
            return list(row[2 + 48 : 2 + 60])
    raise RuntimeError(f'BREPUUS row not found in {p}')

steo_jan22 = _load_steo_brent_2022('jan22')
steo_feb22 = _load_steo_brent_2022('feb22')

T0 = pd.Timestamp('2022-02-24')
TEND = pd.Timestamp('2022-09-30')

# Use one fit's index for the treatment day-set (actual trading days, holiday-aware)
_ref = load_fit('russia', 'preferred', 'elastic_net', variant=VARIANT)
treatment_days = np.exp(_ref['actual']).loc[T0:TEND].index

def _step_daily(monthly_2022, index):
    s = pd.Series(index=index, dtype=float)
    for m_idx, m_val in enumerate(monthly_2022, start=1):
        mask = (s.index.year == 2022) & (s.index.month == m_idx)
        s.loc[mask] = m_val
    return s

steo_jan22_daily = _step_daily(steo_jan22, treatment_days)
steo_feb22_daily = _step_daily(steo_feb22, treatment_days)

rows = []
for model in MODELS:
    r = load_fit('russia', 'preferred', model, variant=VARIANT)
    if r is None:
        continue
    synth = np.exp(r['synth']).loc[T0:TEND]
    actual = np.exp(r['actual']).loc[T0:TEND]
    rows.append({
        'model': model,
        'n_days': len(treatment_days),
        'actual_mean': actual.mean(),
        'synth_mean': synth.mean(),
        'steo_jan22_mean': steo_jan22_daily.mean(),
        'steo_feb22_mean': steo_feb22_daily.mean(),
    })
steo_compare = pd.DataFrame(rows)
steo_compare['att_synth_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['synth_mean']) / steo_compare['synth_mean']
steo_compare['att_steo_jan22_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['steo_jan22_mean']) / steo_compare['steo_jan22_mean']
steo_compare['att_steo_feb22_pct'] = 100 * (steo_compare['actual_mean'] - steo_compare['steo_feb22_mean']) / steo_compare['steo_feb22_mean']
steo_compare['synth_minus_steo_feb22'] = steo_compare['synth_mean'] - steo_compare['steo_feb22_mean']

median_synth = steo_compare['synth_mean'].median()
actual_mean = steo_compare['actual_mean'].iloc[0]
steo_jan22_mean = steo_compare['steo_jan22_mean'].iloc[0]
steo_feb22_mean = steo_compare['steo_feb22_mean'].iloc[0]
steo_compare = pd.concat([steo_compare, pd.DataFrame([{
    'model': 'ENSEMBLE_MEDIAN',
    'n_days': int(steo_compare['n_days'].iloc[0]),
    'actual_mean': actual_mean,
    'synth_mean': median_synth,
    'steo_jan22_mean': steo_jan22_mean,
    'steo_feb22_mean': steo_feb22_mean,
    'att_synth_pct': 100 * (actual_mean - median_synth) / median_synth,
    'att_steo_jan22_pct': 100 * (actual_mean - steo_jan22_mean) / steo_jan22_mean,
    'att_steo_feb22_pct': 100 * (actual_mean - steo_feb22_mean) / steo_feb22_mean,
    'synth_minus_steo_feb22': median_synth - steo_feb22_mean,
}])], ignore_index=True)
save_validation_table(steo_compare, 'external_steo_russia')

print(f'Matched treatment day-set: {T0.date()} - {TEND.date()}  ({len(treatment_days)} trading days)')
print(f'EIA STEO Jan-22 (issued ~2022-01-11): Feb-Sep 2022 step-function mean = ${steo_jan22_mean:.2f}/bbl')
print(f'EIA STEO Feb-22 (issued ~2022-02-08): Feb-Sep 2022 step-function mean = ${steo_feb22_mean:.2f}/bbl')
print()
print(steo_compare.round(2).to_string(index=False))

# Overlay plot: actual + ensemble-median synthetic + STEO step functions
actual_full = np.exp(_ref['actual'])
synth_full = np.exp(_ref['synth'])
fig = go.Figure()
fig.add_trace(go.Scatter(x=actual_full.index, y=actual_full.values, mode='lines',
                         name='Observed Brent', line=dict(color='#111111', width=2)))
fig.add_trace(go.Scatter(x=synth_full.index, y=synth_full.values, mode='lines',
                         name='SCM synthetic (Elastic-net = ensemble median)',
                         line=dict(color='#9467bd', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=steo_feb22_daily.index, y=steo_feb22_daily.values,
                         mode='lines', line_shape='hv',
                         name='EIA STEO Feb-22 (last pre-invasion)',
                         line=dict(color='#2ca02c', width=1.5, dash='dot')))
fig.add_trace(go.Scatter(x=steo_jan22_daily.index, y=steo_jan22_daily.values,
                         mode='lines', line_shape='hv',
                         name='EIA STEO Jan-22',
                         line=dict(color='#ff7f0e', width=1, dash='dot')))
fig.add_vline(x=T0, line_dash='dot', line_color='grey')
fig.add_annotation(x=T0, y=1.02, yref='paper', text='T₀ (invasion)',
                   showarrow=False, font=dict(size=10, color='grey'), xanchor='left')
fig.update_layout(title='Russia 2022 — SCM synthetic vs EIA STEO pre-invasion forecasts',
                  yaxis_title='USD / bbl', xaxis_title='Date',
                  template='plotly_white', hovermode='x unified', height=480,
                  legend=dict(orientation='h', y=-0.18, x=0, xanchor='left', yanchor='top'),
                  margin=dict(t=70, b=80, l=70, r=40))
save_html(fig, 'russia_steo_validation', ROOT / 'plots').show()

Matched treatment day-set: 2022-02-24 - 2022-09-30  (151 trading days)
EIA STEO Jan-22 (issued ~2022-01-11): Feb-Sep 2022 step-function mean = $75.66/bbl
EIA STEO Feb-22 (issued ~2022-02-08): Feb-Sep 2022 step-function mean = $84.94/bbl

          model  n_days  actual_mean  synth_mean  steo_jan22_mean  steo_feb22_mean  att_synth_pct  att_steo_jan22_pct  att_steo_feb22_pct  synth_minus_steo_feb22
     convex_scm     151       108.54       80.82            75.66            84.94          34.29               43.45               27.78                   -4.12
           ascm     151       108.54       97.61            75.66            84.94          11.20               43.45               27.78                   12.67
    elastic_net     151       108.54       88.58            75.66            84.94          22.53               43.45               27.78                    3.64
        xgboost     151       108.54       82.91            75.66            84.94          30.90               43

Last run: 2026-06-09 12:07:23


## Reading the headline result

The headline thesis statement is the **ensemble median for Hormuz** with the IQR as model uncertainty. Russia's ensemble median serves as the **magnitude validation**, anchored two ways:

1. **External**: the SCM ensemble-median counterfactual sits within ~$4/bbl of the EIA STEO Feb-22 step-function forecast over the identical 151-day treatment window (see external-validation section above; ensemble synth $88.92 vs STEO Feb-22 $84.94, Δ +$3.98). The implied ATTs disagree by ~5.7 pp — small enough to read the SCM as internally consistent with an independent structural-model counterfactual issued sixteen days before the invasion.

2. **Internal**: the cross-event weight-transfer table reports whether the Hormuz estimate's confidence should be widened due to regime drift between 2020-22 and 2024-26. Convex SCM and ASCM transfer with plausible pre-RMSPE; the three non-convex models do not — that asymmetry is documented in [validation.md §5f](../docs/validation.md) and inherited by the Hormuz headline.